In [5]:
%pip install matplotlib seaborn pillow pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [6]:
"""
ImageNet Dataset Cleaning and Analysis Pipeline
This notebook analyzes metadata from Visual Layer and creates a cleaned dataset
without modifying the original local ImageNet files.
"""


# Cell 1: Import Libraries and Setup
import json
import os
import shutil
from pathlib import Path
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [7]:
# ============================================================================
# Cell 2: Configuration and Path Setup
# ============================================================================

# Paths
LOCAL_IMAGENET_BASE = Path("/Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC")
METADATA_FILES = [
    Path("/Users/saeedarellano/Desktop/imagenet_jsons/metadata_2.json"),
    Path("/Users/saeedarellano/Desktop/imagenet_jsons/metadata-3.json")
]
OUTPUT_BASE = Path("/Users/saeedarellano/visual_layer/capstone_project_Visual-Layer/clean")
DRY_RUN_OUTPUT = OUTPUT_BASE / "dry_run"
FINAL_OUTPUT = OUTPUT_BASE / "final_cleaned_dataset"

print("Configuration loaded successfully!")
print(f"Local ImageNet base: {LOCAL_IMAGENET_BASE}")
print(f"Output directory: {OUTPUT_BASE}")


Configuration loaded successfully!
Local ImageNet base: /Users/saeedarellano/Desktop/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC
Output directory: /Users/saeedarellano/visual_layer/capstone_project_Visual-Layer/clean


In [8]:
# ============================================================================
# Cell 3: Load and Parse Metadata Files
# ============================================================================

def load_metadata(metadata_path):
    """Load metadata JSON file"""
    with open(metadata_path, 'r') as f:
        return json.load(f)

def extract_user_tags(media_item):
    """Extract user tags from media item"""
    tags = []
    for metadata in media_item.get('metadata_items', []):
        if metadata.get('type') == 'user_tag':
            tags.append(metadata['properties']['tag_name'])
    return tags

def extract_issues(media_item):
    """Extract all issues from media item"""
    issues = []
    for metadata in media_item.get('metadata_items', []):
        if metadata.get('type') == 'issue':
            issues.append({
                'issue_type': metadata['properties']['issue_type'],
                'confidence': metadata['properties'].get('confidence', 0)
            })
    return issues

# Load both metadata files
print("Loading metadata files...")
metadata_2 = load_metadata(METADATA_FILES[0])
metadata_3 = load_metadata(METADATA_FILES[1])

print(f"\nMetadata 2: {metadata_2['info']['total_media_items']} items")
print(f"Metadata 3: {metadata_3['info']['total_media_items']} items")

# Combine all media items
all_media_items = metadata_2['media_items'] + metadata_3['media_items']
print(f"Total media items: {len(all_media_items)}")

Loading metadata files...

Metadata 2: 214 items
Metadata 3: 23 items
Total media items: 237


In [9]:
#============================================================================
# Cell 4: Analyze Local Dataset
# ============================================================================

def scan_local_dataset(base_path, splits=['train', 'val', 'test']):
    """Scan local ImageNet dataset and return file inventory"""
    local_files = {}
    
    for split in splits:
        split_path = base_path / split
        if not split_path.exists():
            print(f"Warning: {split} directory not found")
            continue
        
        local_files[split] = []
        for class_dir in split_path.iterdir():
            if class_dir.is_dir():
                for img_file in class_dir.glob('*.JPEG'):
                    local_files[split].append({
                        'filename': img_file.name,
                        'class_id': class_dir.name,
                        'full_path': img_file
                    })
    
    return local_files

print("Scanning local dataset...")
local_inventory = scan_local_dataset(LOCAL_IMAGENET_BASE)

for split, files in local_inventory.items():
    print(f"{split}: {len(files)} images")

Scanning local dataset...
train: 655570 images
val: 0 images
test: 0 images


In [10]:
def match_files(media_items, local_inventory):
    """Match metadata filenames with local dataset files"""
    matches = {'metadata_2': [], 'metadata_3': [], 'unmatched': []}
    
    # Create lookup dictionary for local files
    local_lookup = {}
    for split, files in local_inventory.items():
        for file_info in files:
            local_lookup[file_info['filename']] = file_info
    
    # Match each media item
    for item in media_items:
        filename = item['file_name']
        source = 'metadata_2' if item in metadata_2['media_items'] else 'metadata_3'
        
        if filename in local_lookup:
            matches[source].append({
                'media_item': item,
                'local_file': local_lookup[filename],
                'filename': filename
            })
        else:
            matches['unmatched'].append({
                'media_item': item,
                'filename': filename,
                'source': source
            })
    
    return matches

print("Matching metadata with local files...")
file_matches = match_files(all_media_items, local_inventory)

print(f"\nMatches from metadata_2: {len(file_matches['metadata_2'])}")
print(f"Matches from metadata_3: {len(file_matches['metadata_3'])}")
print(f"Unmatched items: {len(file_matches['unmatched'])}")

Matching metadata with local files...

Matches from metadata_2: 47
Matches from metadata_3: 7
Unmatched items: 183


In [11]:
# ============================================================================
# Cell 6: Extract User Tags by Category
# ============================================================================

# Define user tag categories we're looking for
TARGET_TAGS = [
    'Mislabel_duplicate',
    'Mislabel_noprimary',
    'train_leakage',
    'Mislabel_nondominant',
    'Mislabel_wrong',
    'Train_leakage'
]

tagged_images = defaultdict(list)

for item in all_media_items:
    tags = extract_user_tags(item)
    for tag in tags:
        # Check for target tags (case-insensitive)
        for target_tag in TARGET_TAGS:
            if target_tag.lower() in tag.lower():
                tagged_images[target_tag].append(item)

print("\nUser Tag Summary:")
for tag, items in tagged_images.items():
    print(f"{tag}: {len(items)} images")


User Tag Summary:
Mislabel_noprimary: 11 images
Mislabel_nondominant: 30 images
Mislabel_wrong: 5 images


In [12]:
def should_remove_image(media_item):
    """
    Determine if an image should be removed (blacklisted) based on user tags.
    Returns: (should_remove: bool, reason: str)
    """
    tags = extract_user_tags(media_item)
    
    # Blacklist criteria - remove images with these tags
    blacklist_tags = [
        'mislabel_duplicate',
        'mislabel_noprimary',
        'train_leakage',
        'mislabel_nondominant',
        'mislabel_wrong'
    ]
    
    for tag in tags:
        for blacklist_tag in blacklist_tags:
            if blacklist_tag.lower() in tag.lower():
                return True, tag
    
    return False, None

# Analyze what will be removed
print("Analyzing blacklist...")
blacklist_summary = {
    'total_images': 0,
    'images_to_remove': 0,
    'images_to_keep': 0,
    'removed_by_tag': Counter(),
    'removed_items': []
}

for source in ['metadata_2', 'metadata_3']:
    for match in file_matches[source]:
        blacklist_summary['total_images'] += 1
        should_remove, reason = should_remove_image(match['media_item'])
        
        if should_remove:
            blacklist_summary['images_to_remove'] += 1
            blacklist_summary['removed_by_tag'][reason] += 1
            blacklist_summary['removed_items'].append({
                'filename': match['filename'],
                'class_id': match['local_file']['class_id'],
                'reason': reason,
                'source': source,
                'media_item': match['media_item']
            })
        else:
            blacklist_summary['images_to_keep'] += 1

print(f"\n{'='*70}")
print("BLACKLIST ANALYSIS")
print(f"{'='*70}")
print(f"Total matched images: {blacklist_summary['total_images']}")
print(f"Images to KEEP: {blacklist_summary['images_to_keep']}")
print(f"Images to REMOVE (blacklisted): {blacklist_summary['images_to_remove']}")
print(f"\nRemoval breakdown by tag:")
for tag, count in blacklist_summary['removed_by_tag'].most_common():
    print(f"  - {tag}: {count} images")

Analyzing blacklist...

BLACKLIST ANALYSIS
Total matched images: 54
Images to KEEP: 44
Images to REMOVE (blacklisted): 10

Removal breakdown by tag:
  - mislabel_nondominant: 6 images
  - mislabel_noprimary: 3 images
  - mislabel_wrong: 1 images


In [13]:
def should_remove_image(media_item):
    """
    Determine if an image should be removed (blacklisted) based on user tags.
    Returns: (should_remove: bool, reason: str)
    """
    tags = extract_user_tags(media_item)
    
    # Blacklist criteria - remove images with these tags
    blacklist_tags = [
        'mislabel_duplicate',
        'mislabel_noprimary',
        'train_leakage',
        'mislabel_nondominant',
        'mislabel_wrong'
    ]
    
    for tag in tags:
        for blacklist_tag in blacklist_tags:
            if blacklist_tag.lower() in tag.lower():
                return True, tag
    
    return False, None

# Analyze what will be removed
print("Analyzing blacklist...")
blacklist_summary = {
    'total_images': 0,
    'images_to_remove': 0,
    'images_to_keep': 0,
    'removed_by_tag': Counter(),
    'removed_items': []
}

for source in ['metadata_2', 'metadata_3']:
    for match in file_matches[source]:
        blacklist_summary['total_images'] += 1
        should_remove, reason = should_remove_image(match['media_item'])
        
        if should_remove:
            blacklist_summary['images_to_remove'] += 1
            blacklist_summary['removed_by_tag'][reason] += 1
            blacklist_summary['removed_items'].append({
                'filename': match['filename'],
                'class_id': match['local_file']['class_id'],
                'reason': reason,
                'source': source,
                'media_item': match['media_item']
            })
        else:
            blacklist_summary['images_to_keep'] += 1

print(f"\n{'='*70}")
print("BLACKLIST ANALYSIS")
print(f"{'='*70}")
print(f"Total matched images: {blacklist_summary['total_images']}")
print(f"Images to KEEP: {blacklist_summary['images_to_keep']}")
print(f"Images to REMOVE (blacklisted): {blacklist_summary['images_to_remove']}")
print(f"\nRemoval breakdown by tag:")
for tag, count in blacklist_summary['removed_by_tag'].most_common():
    print(f"  - {tag}: {count} images")

Analyzing blacklist...

BLACKLIST ANALYSIS
Total matched images: 54
Images to KEEP: 44
Images to REMOVE (blacklisted): 10

Removal breakdown by tag:
  - mislabel_nondominant: 6 images
  - mislabel_noprimary: 3 images
  - mislabel_wrong: 1 images
